<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4;">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/media-processing-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="display: flex; justify-content: flex-end; flex: 1; gap: 8px; align-items: center; padding: 0;">
        <a href="https://opencv.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/opencv/logo/OpenCV_logo_no_text-1.svg"
                 alt="OpenCV Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://pillow.readthedocs.io/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/pillow/logo/pillow-logo-248x250.png"
                 alt="PIL Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://scikit-image.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/scikit-image/logo/logo.png"
                 alt="scikit-image Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://scipy.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/scipy/logo/logo.svg"
                 alt="SciPy Logo"
                 style="max-height: 48px; width: auto;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Load Images](#toc2_)    
- [Image Interpolation](#toc3_)    
  - [Interpolation](#toc3_1_)    
    - [Using OpenCV](#toc3_1_1_)    
    - [Using PIL](#toc3_1_2_)    
    - [Using scikit-image](#toc3_1_3_)    
    - [Using Scipy](#toc3_1_4_)    
  - [Aspect Ratio](#toc3_2_)    
    - [Keep Aspect](#toc3_2_1_)    
    - [Distort to Fill](#toc3_2_2_)    
  - [Common Interpolations](#toc3_3_)    
    - [Nearest Neighbor](#toc3_3_1_)    
      - [Manual](#toc3_3_1_1_)    
      - [Using OpenCV](#toc3_3_1_2_)    
    - [Linear](#toc3_3_2_)    
      - [Manual](#toc3_3_2_1_)    
      - [Using OpenCV](#toc3_3_2_2_)    
    - [Cubic](#toc3_3_3_)    
    - [Area](#toc3_3_4_)    
    - [Lanczos](#toc3_3_5_)    
  - [Interpolation using Fourier Transform](#toc3_4_)    
  - [Comparison](#toc3_5_)    
    - [Down Scaling](#toc3_5_1_)    
    - [Up Scaling](#toc3_5_2_)    
  - [Learnable / AI-based Interpolation](#toc3_6_)    
    - [Super-Resolution with CNNs](#toc3_6_1_)    
    - [Super-Resolution with GANs](#toc3_6_2_)    
    - [Fourier / Frequency Domain Super-Resolution](#toc3_6_3_)    
    - [Learned Upsampling Layers in CNNs](#toc3_6_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [ ]:
from pathlib import Path

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
import skimage as ski
from matplotlib.gridspec import GridSpec
from numpy.typing import NDArray
from PIL import Image

In [ ]:
# disable automatic figure display (plt.show() required)
# this ensures consistency with .py scripts and gives full control over when plots appear
plt.ioff()

# <a id='toc2_'></a>[Load Images](#toc0_)


In [ ]:
imp_1 = Path("../assets/external/wikipedia/Guernica_(Picasso)/PicassoGuernica.jpg")
imp_2 = Path(
    "../assets/external/wikipedia/The_Tower_of_Babel_(Bruegel)/Pieter_Bruegel_the_Elder_-_The_Tower_of_Babel_(Vienna)_-_Google_Art_Project_-_edited.jpg"
)

In [ ]:
img_1 = cv2.imread(imp_1, flags=cv2.IMREAD_GRAYSCALE)
img_2 = cv2.imread(imp_2, flags=cv2.IMREAD_COLOR_RGB)

In [ ]:
img_1_pil = Image.fromarray(img_1)
img_2_pil = Image.fromarray(img_2)

In [ ]:
# plot
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(20, 8), layout="constrained")
axs[0].imshow(img_1, cmap="gray", vmin=0, vmax=255)
axs[0].set(title=imp_1.name, xticks=[], yticks=[])
axs[1].imshow(img_2, vmin=0, vmax=255)
axs[1].set(title=imp_2.name, xticks=[], yticks=[])
plt.show()

# <a id='toc3_'></a>[Image Interpolation](#toc0_)

It refers to the **guess** of intensity values at missing locations When **resizing** an image.

$$
\mathbf{S}=
\begin{bmatrix}
S_x\\
S_y
\end{bmatrix},
\qquad
S_x=\frac{X_{\text{new}}}{X_{\text{old}}},
\qquad
S_y=\frac{Y_{\text{new}}}{Y_{\text{old}}}.
$$

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/original/interpolation/interpolation-example-1.svg" alt="interpolation-example-1.svg" 
         style="min-width: 512px; width: 60%; height: auto; border-radius: 16px;">
    <p><em>Figure 1: Nearest Neighbor Interpolation + Preserve Aspect Ratio</em></p>
</div>

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/original/interpolation/interpolation-example-2.svg" alt="interpolation-example-2.svg" 
         style="min-width: 512px; width: 60%; height: auto; border-radius: 16px;">
    <p><em>Figure 2: Nearest Neighbor Interpolation + Ignore Aspect Ratio</em></p>
</div>

**Interpolation Methods:**

| Interpolation Method | Downscaling Quality | Upscaling Quality | Performance   |
| -------------------- | ------------------- | ----------------- | ------------- |
| Nearest Neighbor     | | | ⭐⭐⭐⭐⭐ |
| Box                  | ⭐ | | ⭐⭐⭐⭐ |
| Bilinear             | ⭐ | ⭐ | ⭐⭐⭐ |
| Hamming              | ⭐⭐ | | ⭐⭐⭐ |
| Bicubic              | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| Lanczos              | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐ |

Reference: [pillow.readthedocs.io/en/stable/handbook/concepts.html#filters-comparison-table](https://pillow.readthedocs.io/en/stable/handbook/concepts.html#filters-comparison-table)


## <a id='toc3_1_'></a>[Interpolation](#toc0_)


### <a id='toc3_1_1_'></a>[Using OpenCV](#toc0_)

✍️ **Common OpenCV interpolation flags:**

- `cv2.INTER_NEAREST`
- `cv2.INTER_LINEAR`
- `cv2.INTER_CUBIC`
- `cv2.INTER_AREA`
- `cv2.INTER_LANCZOS4`


📝 **Docs**:

- `cv2.resize`: [docs.opencv.org/master/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d](https://docs.opencv.org/master/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d)
- `InterpolationFlags`: [docs.opencv.org/master/da/d54/group__imgproc__transform.html#gga5bb5a1fea74ea38e1a5445ca803ff121ac97d8e4880d8b5d509e96825c7522deb](https://docs.opencv.org/master/da/d54/group__imgproc__transform.html#gga5bb5a1fea74ea38e1a5445ca803ff121ac97d8e4880d8b5d509e96825c7522deb)


In [ ]:
fx_1 = fy_1 = 0.5
img_1_ds_ocv = cv2.resize(img_1, None, fx=fx_1, fy=fy_1, interpolation=cv2.INTER_CUBIC)

In [ ]:
fx_2 = fy_2 = 2.0
img_1_us_ocv = cv2.resize(img_1, None, fx=fx_2, fy=fy_2, interpolation=cv2.INTER_CUBIC)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_ds_ocv, img_1, img_1_us_ocv]
titles = [f"fx: {fx_1}x, fy: {fy_1}x", "Original", f"fx: {fx_2}x, fy: {fy_2}x"]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(ncols):
    axs[0, i].imshow(images[i], cmap="gray", interpolation="none")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(images[i][roi[i][0], roi[i][1]], cmap="gray", interpolation="none")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

### <a id='toc3_1_2_'></a>[Using PIL](#toc0_)

✍️ **Common PIL resampling flags:**

- `Image.Resampling.NEAREST`
- `Image.Resampling.BILINEAR`
- `Image.Resampling.BICUBIC`
- `Image.Resampling.LANCZOS`
- `Image.Resampling.HAMMING`
- `Image.Resampling.BOX`

📝 **Docs**:

- `PIL.Image.Image.resize`: [pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.resize](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.resize)
- Filters: [pillow.readthedocs.io/en/stable/handbook/concepts.html#concept-filters](https://pillow.readthedocs.io/en/stable/handbook/concepts.html#concept-filters)


In [ ]:
fx_1 = fy_1 = 0.5
sx_1 = int(img_1_pil.size[0] * fx_1)
sy_1 = int(img_1_pil.size[1] * fy_1)
img_1_ds_pil = img_1_pil.resize((sx_1, sy_1), resample=Image.Resampling.BICUBIC)

In [ ]:
fx_2 = fy_2 = 2.0
sx_2 = int(img_1_pil.size[0] * fx_2)
sy_2 = int(img_1_pil.size[1] * fy_2)
img_1_us_pil = img_1_pil.resize((sx_2, sy_2), resample=Image.Resampling.BICUBIC)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_ds_pil, img_1, img_1_us_pil]
titles = [img_1_ds_pil.size, f"{img_1_pil.size} [Original]", img_1_us_pil.size]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(len(images)):
    axs[0, i].imshow(images[i], cmap="gray", interpolation="none")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(np.asarray(images[i])[roi[i][0], roi[i][1]], cmap="gray", interpolation="none")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

### <a id='toc3_1_3_'></a>[Using scikit-image](#toc0_)

✍️ **Common scikit-image order flags:**

- `0: Nearest-neighbor`
- `1: Bi-linear (default)`
- `2: Bi-quadratic`
- `3: Bi-cubic`
- `4: Bi-quartic`
- `5: Bi-quintic`

📝 **Docs**:

- `skimage.transform.resize`: [scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.resize](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.resize)
- `skimage.transform.warp`: [scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.warp](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.warp)


In [ ]:
fx_1 = fy_1 = 0.5
sx_1 = int(img_1.shape[1] * fx_1)
sy_1 = int(img_1.shape[0] * fy_1)
img_1_ds_ski = ski.transform.resize(img_1, (sy_1, sx_1), anti_aliasing=False)

In [ ]:
fx_2 = fy_2 = 2.0
sx_1 = int(img_1.shape[1] * fx_2)
sy_1 = int(img_1.shape[0] * fy_2)
img_1_us_ski = ski.transform.resize(img_1, (sy_2, sx_2), anti_aliasing=False)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_ds_ski, img_1, img_1_us_ski]
titles = [img_1_ds_ski.shape, f"{img_1.shape} [Original]", img_1_us_ski.shape]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(len(images)):
    axs[0, i].imshow(images[i], cmap="gray")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(images[i][roi[i][0], roi[i][1]], cmap="gray")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

### <a id='toc3_1_4_'></a>[Using Scipy](#toc0_)

✍️ **Common Scipy order flags:**

- `0: Nearest-neighbor`
- `1: Bi-linear (default)`
- `2: Bi-quadratic`
- `3: Bi-cubic`
- `4: Bi-quartic`
- `5: Bi-quintic`

📝 **Docs**:

- `scipy.ndimage.zoom`: [docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.zoom.html#scipy.ndimage.zoom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.zoom.html#scipy.ndimage.zoom)


In [ ]:
fx_1 = fy_1 = 0.5
img_1_ds_scp = scipy.ndimage.zoom(img_1, zoom=(fy_1, fx_1), order=3)

In [ ]:
fx_2 = fy_2 = 2.0
img_1_us_scp = scipy.ndimage.zoom(img_1, zoom=(fy_2, fx_2), order=3)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_ds_scp, img_1, img_1_us_scp]
titles = [f"fx: {fx_1}x, fy: {fy_1}x", "Original", f"fx: {fx_2}x, fy: {fy_2}x"]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(len(images)):
    axs[0, i].imshow(images[i], cmap="gray", interpolation="none")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(images[i][roi[i][0], roi[i][1]], cmap="gray", interpolation="none")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

## <a id='toc3_2_'></a>[Aspect Ratio](#toc0_)


### <a id='toc3_2_1_'></a>[Keep Aspect](#toc0_)


In [ ]:
fx_1 = fy_1 = 0.5
img_1_ka_1 = cv2.resize(img_1, None, fx=fx_1, fy=fy_1, interpolation=cv2.INTER_CUBIC)

In [ ]:
fx_2 = fy_2 = 2.0
img_1_ka_2 = cv2.resize(img_1, None, fx=fx_2, fy=fy_2, interpolation=cv2.INTER_CUBIC)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_ka_1, img_1, img_1_ka_2]
titles = [f"fx: {fx_1}x, fy: {fy_1}x", "Original", f"fx: {fx_2}x, fy: {fy_2}x"]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(len(images)):
    axs[0, i].imshow(images[i], cmap="gray", interpolation="none")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(images[i][roi[i][0], roi[i][1]], cmap="gray", interpolation="none")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

### <a id='toc3_2_2_'></a>[Distort to Fill](#toc0_)


In [ ]:
fx_1 = 1.0
fy_1 = 0.5
img_1_df_1 = cv2.resize(img_1, None, fx=fx_1, fy=fy_1, interpolation=cv2.INTER_CUBIC)

In [ ]:
fx_2 = 1.5
fy_2 = 2.0
img_1_df_2 = cv2.resize(img_1, None, fx=fx_2, fy=fy_2, interpolation=cv2.INTER_CUBIC)

In [ ]:
cow_r, cow_c = slice(0, 75), slice(40, 115)
images = [img_1_df_1, img_1, img_1_df_2]
titles = [f"fx: {fx_1}x, fy: {fy_1}x", "Original", f"fx: {fx_2}x, fy: {fy_2}x"]
roi = [
    (slice(int(cow_r.start * fy_1), int(cow_r.stop * fy_1)), slice(int(cow_c.start * fx_1), int(cow_c.stop * fx_1))),
    (cow_r, cow_c),
    (slice(int(cow_r.start * fy_2), int(cow_r.stop * fy_2)), slice(int(cow_c.start * fx_2), int(cow_c.stop * fx_2))),
]

# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), layout="compressed")
for i in range(len(images)):
    axs[0, i].imshow(images[i], cmap="gray", interpolation="none")
    axs[0, i].add_patch(
        patches.Rectangle(
            (roi[i][1].start, roi[i][0].start),
            roi[i][1].stop - roi[i][1].start,
            roi[i][0].stop - roi[i][0].start,
            linewidth=2,
            edgecolor="r",
            facecolor="none",
        )
    )
    axs[0, i].set_title(titles[i])
    axs[1, i].imshow(images[i][roi[i][0], roi[i][1]], cmap="gray", interpolation="none")
    axs[1, i].set_title(f"{titles[i]} [zoom]")

plt.show()

## <a id='toc3_3_'></a>[Common Interpolations](#toc0_)


### <a id='toc3_3_1_'></a>[Nearest Neighbor](#toc0_)

- For **nearest-neighbor** interpolation, the output pixel value is taken from the **nearest source pixel** (no blending).

🔢 **Formula:**

- Let the source image be $I(u,v)$ and the resized output be $J(x,y)$.
- If the output pixel $(x,y)$ maps to continuous source coordinates $(u,v)$, then:

$$
u=\frac{x}{S_x},\qquad v=\frac{y}{S_y}
$$

$$
J(x,y)= I\!\left(\operatorname{round}(u),\ \operatorname{round}(v)\right)
$$


#### <a id='toc3_3_1_1_'></a>[Manual](#toc0_)


In [ ]:
def nearest_interpolation(image: NDArray, new_height: int, new_width: int, mode: str = "efficient") -> NDArray:
    """
    Nearest-neighbor interpolation consistent with:

        u = x / Sx
        v = y / Sy
        J(x,y) = I(round(u), round(v))

    Here:
      - x = output column index
      - y = output row index
      - Sx = new_width / width
      - Sy = new_height / height
    """
    height, width = image.shape[:2]
    Sx = new_width / width
    Sy = new_height / height

    if mode == "inefficient":
        new_image = np.zeros((new_height, new_width, *image.shape[2:]), dtype=image.dtype)
        for y in range(new_height):  # output row index
            for x in range(new_width):  # output column index
                u = x / Sx
                v = y / Sy
                src_x = np.clip(round(u), 0, width - 1)
                src_y = np.clip(round(v), 0, height - 1)
                new_image[y, x] = image[src_y, src_x]

    elif mode == "efficient":
        x = np.arange(new_width, dtype=np.float64)  # output columns
        y = np.arange(new_height, dtype=np.float64)  # output rows

        u = x / Sx
        v = y / Sy

        x_indices = np.clip(np.round(u).astype(int), 0, width - 1)
        y_indices = np.clip(np.round(v).astype(int), 0, height - 1)

        new_image = image[y_indices[:, np.newaxis], x_indices]

    else:
        raise ValueError(f"Unknown mode '{mode}'. Choose 'efficient' or 'inefficient'.")

    return new_image.astype(image.dtype)

#### <a id='toc3_3_1_2_'></a>[Using OpenCV](#toc0_)


In [ ]:
img_2_nni_1 = cv2.resize(img_2, dsize=(492, 360), interpolation=cv2.INTER_NEAREST_EXACT)
img_2_nni_2 = cv2.resize(img_2, dsize=(1968, 1440), interpolation=cv2.INTER_NEAREST_EXACT)
img_2_nni_3 = cv2.resize(img_2, dsize=(492, 720), interpolation=cv2.INTER_NEAREST_EXACT)
img_2_nni_4 = cv2.resize(img_2, dsize=(1476, 720), interpolation=cv2.INTER_NEAREST_EXACT)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_nni_1, img_2_nni_2, img_2_nni_3, img_2_nni_4]
titles = [f"{img_2.shape} [Original]", img_2_nni_1.shape, img_2_nni_2.shape, img_2_nni_3.shape, img_2_nni_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

### <a id='toc3_3_2_'></a>[Linear](#toc0_)

- For **linear interpolation**, the output pixel value is computed by taking a **weighted average of the 4 nearest source pixels** around $(u,v)$.

🔢 **Formula:**

- Let the source image be $I(u,v)$ and the resized output be $J(x,y)$.
- If the output pixel $(x,y)$ maps to continuous source coordinates $(u,v)$, then:

$$
u=\frac{x}{S_x},\qquad v=\frac{y}{S_y}
$$

$$
u_0=\left\lfloor u\right\rfloor,\qquad v_0=\left\lfloor v\right\rfloor
$$

$$
J(x,y)=\sum_{m=0}^{1}\sum_{n=0}^{1} I\!\big(u_0+m,\ v_0+n\big)\,\big(1-|u-(u_0+m)|\big)\,\big(1-|v-(v_0+n)|\big)
$$


#### <a id='toc3_3_2_1_'></a>[Manual](#toc0_)


In [ ]:
def bilinear_interpolation(image: NDArray, new_height: int, new_width: int) -> NDArray:
    """
    Vectorized bilinear interpolation consistent with:
      u = x / Sx, v = y / Sy
      u0=floor(u), v0=floor(v)
      J(x,y)= sum_{m=0..1} sum_{n=0..1} I(u0+m, v0+n) *
              (1-|u-(u0+m)|) * (1-|v-(v0+n)|)
    """
    H, W = image.shape[:2]
    has_channels = image.ndim == 3
    C = image.shape[2] if has_channels else 1

    Sx = new_width / W
    Sy = new_height / H

    # output grid (x: cols, y: rows)
    x = np.arange(new_width, dtype=np.float64)  # output columns
    y = np.arange(new_height, dtype=np.float64)  # output rows

    # u,v for every output pixel
    u = x[None, :] / Sx  # shape (1, new_width)
    v = y[:, None] / Sy  # shape (new_height, 1)

    # base indices u0, v0
    u0 = np.floor(u).astype(np.int64)
    v0 = np.floor(v).astype(np.int64)

    # Clamp so u0+1 and v0+1 are valid
    u0 = np.clip(u0, 0, W - 2)
    v0 = np.clip(v0, 0, H - 2)

    # fractional coords
    # since u0=floor(u), du in [0,1). Similarly dv.
    du = u - u0
    dv = v - v0

    # weights (matches 1 - |u-(u0+m)|)
    w00 = (1.0 - du) * (1.0 - dv)  # (u0,   v0)
    w10 = du * (1.0 - dv)  # (u0+1, v0)
    w01 = (1.0 - du) * dv  # (u0,   v0+1)
    w11 = du * dv  # (u0+1, v0+1)

    if has_channels:
        # gather pixels for 4 neighbors using broadcasting
        # image[v, u] for each output pixel: need v index array shape (new_height,new_width)
        v0b = v0
        u0b = u0

        # (new_h, new_w, C)
        I00 = image[v0b, u0b]
        I10 = image[v0b, u0b + 1]
        I01 = image[v0b + 1, u0b]
        I11 = image[v0b + 1, u0b + 1]

        out = w00[..., None] * I00 + w10[..., None] * I10 + w01[..., None] * I01 + w11[..., None] * I11
    else:
        I00 = image[v0, u0]
        I10 = image[v0, u0 + 1]
        I01 = image[v0 + 1, u0]
        I11 = image[v0 + 1, u0 + 1]

        out = w00 * I00 + w10 * I10 + w01 * I01 + w11 * I11

    # cast back to original dtype (with clipping for integer types)
    if np.issubdtype(image.dtype, np.integer):
        info = np.iinfo(image.dtype)
        out = np.clip(out, info.min, info.max)

    return out.astype(image.dtype)

#### <a id='toc3_3_2_2_'></a>[Using OpenCV](#toc0_)


In [ ]:
img_2_bli_1 = cv2.resize(img_2, dsize=(492, 360), interpolation=cv2.INTER_LINEAR)
img_2_bli_2 = cv2.resize(img_2, dsize=(1968, 1440), interpolation=cv2.INTER_LINEAR)
img_2_bli_3 = cv2.resize(img_2, dsize=(492, 720), interpolation=cv2.INTER_LINEAR)
img_2_bli_4 = cv2.resize(img_2, dsize=(1476, 720), interpolation=cv2.INTER_LINEAR)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_bli_1, img_2_bli_2, img_2_bli_3, img_2_bli_4]
titles = [f"{img_2.shape} [Original]", img_2_bli_1.shape, img_2_bli_2.shape, img_2_bli_3.shape, img_2_bli_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

### <a id='toc3_3_3_'></a>[Cubic](#toc0_)

- For **cubic interpolation**, the output pixel value is computed using a **weighted combination of 16 nearest source samples** (4×4 neighborhood) around the continuous source coordinate $(u,v)$.

🔢 **Formula (cubic convolution / bicubic):**

- Let the source image be $I(u,v)$ and the resized output be $J(x,y)$.
- If the output pixel $(x,y)$ maps to continuous source coordinates $(u,v)$:

$$
u=\frac{x}{S_x},\qquad v=\frac{y}{S_y}
$$

- Let the integer base coordinates be:

$$
u_0=\left\lfloor u\right\rfloor,\qquad v_0=\left\lfloor v\right\rfloor
$$

- Then bicubic interpolation is:

$$
J(x,y)=\sum_{m=-1}^{2}\sum_{n=-1}^{2}
I\!\big(u_0+m,\ v_0+n\big)\;
w(u-(u_0+m))\;
w(v-(v_0+n))
$$

- Where the cubic kernel $w(t)$ (cubic convolution) is typically defined as:

$$
w(t)=
\begin{cases}
( a+2)|t|^3-( a+3)|t|^2+1, & |t|<1 \\
a|t|^3-5a|t|^2+8a|t|-4a, & 1\le |t|<2 \\
0, & |t|\ge 2
\end{cases}
$$

- Common choice: $a=-0.5$ (Catmull–Rom) or $a=-0.75$ (Keys) depending on the desired sharpness.


In [ ]:
img_2_bci_1 = cv2.resize(img_2, dsize=(492, 360), interpolation=cv2.INTER_CUBIC)
img_2_bci_2 = cv2.resize(img_2, dsize=(1968, 1440), interpolation=cv2.INTER_CUBIC)
img_2_bci_3 = cv2.resize(img_2, dsize=(492, 720), interpolation=cv2.INTER_CUBIC)
img_2_bci_4 = cv2.resize(img_2, dsize=(1476, 720), interpolation=cv2.INTER_CUBIC)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_bci_1, img_2_bci_2, img_2_bci_3, img_2_bci_4]
titles = [f"{img_2.shape} [Original]", img_2_bci_1.shape, img_2_bci_2.shape, img_2_bci_3.shape, img_2_bci_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

### <a id='toc3_3_4_'></a>[Area](#toc0_)

- For **area interpolation** (a.k.a. resampling by **pixel-area averaging**), the output pixel is computed as the **average of the source pixel values weighted by the overlap area** between:
  - the output pixel footprint in the source image coordinates, and
  - each source pixel footprint.

🔢 **Formula (conceptual area/overlap weighting):**

- Let the source image be $I(u,v)$ defined on source pixel cells, and the resized output be $J(x,y)$.
- Map the output pixel center $(x,y)$ to continuous source coordinates $(u,v)$ using your chosen scale factors $S_x, S_y$ (or equivalently map pixel boundaries).
- Define the continuous **source-coordinate rectangle** covered by output pixel $(x,y)$ as:
  - $[u_{\min},u_{\max}) \times [v_{\min},v_{\max})$.
- Let source pixel indices $(i,j)$ correspond to unit-area cells:
  - $[i,i+1)\times[j,j+1)$.
- Then:
$$
J(x,y)=\frac{
\sum_{i}\sum_{j} I(i,j)\; A_{ij}
}{
\sum_{i}\sum_{j} A_{ij}
}
$$

- Where $A_{ij}$ is the **overlap area** between the output pixel rectangle and the source pixel cell $(i,j)$:
$$
A_{ij} =
\Big(\max(0,\min(u_{\max},i+1)-\max(u_{\min},i))\Big)\;
\Big(\max(0,\min(v_{\max},j+1)-\max(v_{\min},j))\Big)
$$

- Note: The denominator normalizes by the total covered area (often it equals the output pixel area if the mapping is consistent and fully covered).


In [ ]:
img_2_ai_1 = cv2.resize(img_2, dsize=(492, 360), interpolation=cv2.INTER_AREA)
img_2_ai_2 = cv2.resize(img_2, dsize=(1968, 1440), interpolation=cv2.INTER_AREA)
img_2_ai_3 = cv2.resize(img_2, dsize=(492, 720), interpolation=cv2.INTER_AREA)
img_2_ai_4 = cv2.resize(img_2, dsize=(1476, 720), interpolation=cv2.INTER_AREA)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_ai_1, img_2_ai_2, img_2_ai_3, img_2_ai_4]
titles = [f"{img_2.shape} [Original]", img_2_ai_1.shape, img_2_ai_2.shape, img_2_ai_3.shape, img_2_ai_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

### <a id='toc3_3_5_'></a>[Lanczos](#toc0_)

- For **Lanczos interpolation**, the output pixel is computed using a **windowed sinc kernel** over a finite neighborhood around the continuous source coordinate $(u,v)$.

🔢 **Formula (Lanczos-k bicubic extension):**

- Let the source image be $I(u,v)$ and the resized output be $J(x,y)$.
- Map output pixel $(x,y)$ to continuous source coordinates $(u,v)$:

$$
u=\frac{x}{S_x},\qquad v=\frac{y}{S_y}
$$

- Let the integer base coordinates be:

$$
u_0=\left\lfloor u\right\rfloor,\qquad v_0=\left\lfloor v\right\rfloor
$$

- With Lanczos parameter $k$ (commonly $k=2$ or $k=3$), define the 1D Lanczos kernel:

$$
\operatorname{sinc}(t)=
\begin{cases}
\frac{\sin(\pi t)}{\pi t}, & t\ne 0\\
1, & t=0
\end{cases}
$$

$$
\operatorname{lanczos}_k(t)=
\begin{cases}
\operatorname{sinc}(t)\,\operatorname{sinc}\left(\frac{t}{k}\right), & |t|<k\\
0, & |t|\ge k
\end{cases}
$$

- Then **2D separable Lanczos interpolation** is:

$$
J(x,y)=\sum_{m=-k+1}^{k}\sum_{n=-k+1}^{k}
I\!\big(u_0+m,\ v_0+n\big)\;
\operatorname{lanczos}_k\!\big(u-(u_0+m)\big)\;
\operatorname{lanczos}_k\!\big(v-(v_0+n)\big)
$$


In [ ]:
img_2_l4i_1 = cv2.resize(img_2, dsize=(492, 360), interpolation=cv2.INTER_LANCZOS4)
img_2_l4i_2 = cv2.resize(img_2, dsize=(1968, 1440), interpolation=cv2.INTER_LANCZOS4)
img_2_l4i_3 = cv2.resize(img_2, dsize=(492, 720), interpolation=cv2.INTER_LANCZOS4)
img_2_l4i_4 = cv2.resize(img_2, dsize=(1476, 720), interpolation=cv2.INTER_LANCZOS4)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_l4i_1, img_2_l4i_2, img_2_l4i_3, img_2_l4i_4]
titles = [f"{img_2.shape} [Original]", img_2_l4i_1.shape, img_2_l4i_2.shape, img_2_l4i_3.shape, img_2_l4i_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

## <a id='toc3_4_'></a>[Interpolation using Fourier Transform](#toc0_)


In [ ]:
def fourier_transform_interpolation(image: NDArray, new_height: int, new_width: int) -> NDArray[np.uint8]:
    # convert image to float32 for processing
    img_float = image.astype(np.float32)

    # Handle RGB images (3 channels)
    if len(img_float.shape) == 3:
        # For RGB images, process each channel separately
        result_channels = []
        for channel in range(img_float.shape[2]):
            channel_img = img_float[:, :, channel]
            # perform 2D FFT
            fft_img = np.fft.fft2(channel_img)
            # shift the zero frequency component to the center
            fft_shifted = np.fft.fftshift(fft_img)
            # get original dimensions
            height, width = channel_img.shape
            # create new frequency domain with desired dimensions
            new_fft = np.zeros((new_height, new_width), dtype=complex)
            # calculate the center points
            center_y, center_x = height // 2, width // 2
            new_center_y, new_center_x = new_height // 2, new_width // 2
            # calculate the crop/expand bounds
            crop_height = min(height, new_height)
            crop_width = min(width, new_width)
            # copy the frequency data
            y_start = max(0, new_center_y - crop_height // 2)
            y_end = y_start + crop_height
            x_start = max(0, new_center_x - crop_width // 2)
            x_end = x_start + crop_width
            # handle cases where we're upsampling
            if new_height > height or new_width > width:
                # for upsampling, we need to pad with zeros
                y_start_fft = max(0, center_y - crop_height // 2)
                y_end_fft = y_start_fft + crop_height
                x_start_fft = max(0, center_x - crop_width // 2)
                x_end_fft = x_start_fft + crop_width
                # pad with zeros for upsampling
                new_fft[y_start:y_end, x_start:x_end] = fft_shifted[y_start_fft:y_end_fft, x_start_fft:x_end_fft]
            else:
                # for downsampling, we crop
                y_start_fft = max(0, center_y - crop_height // 2)
                y_end_fft = y_start_fft + crop_height
                x_start_fft = max(0, center_x - crop_width // 2)
                x_end_fft = x_start_fft + crop_width
                # crop the frequency domain
                new_fft[y_start:y_end, x_start:x_end] = fft_shifted[y_start_fft:y_end_fft, x_start_fft:x_end_fft]
            # shift back
            new_fft_unshifted = np.fft.ifftshift(new_fft)
            # perform inverse FFT
            img_back = np.fft.ifft2(new_fft_unshifted)
            # take the real part
            img_real = np.real(img_back)
            # normalize to 0-255 range
            img_normalized = (img_real - img_real.min()) / (img_real.max() - img_real.min()) * 255
            result_channels.append(img_normalized)
        # Stack channels back together
        return np.stack(result_channels, axis=2).astype(np.uint8)
    else:
        # Original code for grayscale images
        # perform 2D FFT
        fft_img = np.fft.fft2(img_float)
        # shift the zero frequency component to the center
        fft_shifted = np.fft.fftshift(fft_img)
        # get original dimensions
        height, width = img_float.shape
        # create new frequency domain with desired dimensions
        new_fft = np.zeros((new_height, new_width), dtype=complex)
        # calculate the center points
        center_y, center_x = height // 2, width // 2
        new_center_y, new_center_x = new_height // 2, new_width // 2
        # calculate the crop/expand bounds
        crop_height = min(height, new_height)
        crop_width = min(width, new_width)
        # copy the frequency data
        y_start = max(0, new_center_y - crop_height // 2)
        y_end = y_start + crop_height
        x_start = max(0, new_center_x - crop_width // 2)
        x_end = x_start + crop_width
        # handle cases where we're upsampling
        if new_height > height or new_width > width:
            # for upsampling, we need to pad with zeros
            y_start_fft = max(0, center_y - crop_height // 2)
            y_end_fft = y_start_fft + crop_height
            x_start_fft = max(0, center_x - crop_width // 2)
            x_end_fft = x_start_fft + crop_width
            # pad with zeros for upsampling
            new_fft[y_start:y_end, x_start:x_end] = fft_shifted[y_start_fft:y_end_fft, x_start_fft:x_end_fft]
        else:
            # for downsampling, we crop
            y_start_fft = max(0, center_y - crop_height // 2)
            y_end_fft = y_start_fft + crop_height
            x_start_fft = max(0, center_x - crop_width // 2)
            x_end_fft = x_start_fft + crop_width
            # crop the frequency domain
            new_fft[y_start:y_end, x_start:x_end] = fft_shifted[y_start_fft:y_end_fft, x_start_fft:x_end_fft]
        # shift back
        new_fft_unshifted = np.fft.ifftshift(new_fft)
        # perform inverse FFT
        img_back = np.fft.ifft2(new_fft_unshifted)
        # take the real part
        img_real = np.real(img_back)
        # normalize to 0-255 range
        img_normalized = (img_real - img_real.min()) / (img_real.max() - img_real.min()) * 255
        # convert back to uint8
        return img_normalized.astype(np.uint8)

In [ ]:
img_2_fti_1 = fourier_transform_interpolation(img_2, 360, 492)
img_2_fti_2 = fourier_transform_interpolation(img_2, 1440, 1968)
img_2_fti_3 = fourier_transform_interpolation(img_2, 720, 492)
img_2_fti_4 = fourier_transform_interpolation(img_2, 720, 1476)

In [ ]:
# plot
fig = plt.figure(figsize=(20, 12), layout="constrained")
gs = GridSpec(nrows=2, ncols=3, figure=fig)
axs = [fig.add_subplot(i) for i in [gs[:, 0], gs[0, 1], gs[0, 2], gs[1, 1], gs[1, 2]]]
images = [img_2, img_2_fti_1, img_2_fti_2, img_2_fti_3, img_2_fti_4]
titles = [f"{img_2.shape} [Original]", img_2_fti_1.shape, img_2_fti_2.shape, img_2_fti_3.shape, img_2_fti_4.shape]
for ax, img, title in zip(axs, images, titles):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
plt.show()

## <a id='toc3_5_'></a>[Comparison](#toc0_)


In [ ]:
img_1_roi = img_1[:, :211]

### <a id='toc3_5_1_'></a>[Down Scaling](#toc0_)


In [ ]:
img_1_nni_ds = cv2.resize(img_1_roi, dsize=(106, 106), interpolation=cv2.INTER_NEAREST_EXACT)
img_1_bli_ds = cv2.resize(img_1_roi, dsize=(106, 106), interpolation=cv2.INTER_LINEAR)
img_1_bci_ds = cv2.resize(img_1_roi, dsize=(106, 106), interpolation=cv2.INTER_CUBIC)
img_1_ai_ds = cv2.resize(img_1_roi, dsize=(106, 106), interpolation=cv2.INTER_AREA)
img_1_l4i_ds = cv2.resize(img_1_roi, dsize=(106, 106), interpolation=cv2.INTER_LANCZOS4)
img_1_fti_ds = fourier_transform_interpolation(img_1_roi, 106, 106)

In [ ]:
titles = [
    f"{img_1.shape} Original",
    f"{img_1_nni_ds.shape} nearest",
    f"{img_1_bli_ds.shape} bilinear",
    f"{img_1_bci_ds.shape} bicubic",
    f"{img_1_ai_ds.shape} area",
    f"{img_1_l4i_ds.shape} lanczos",
    f"{img_1_fti_ds.shape} fourier transform",
]
images = [
    img_1_roi,
    img_1_nni_ds,
    img_1_bli_ds,
    img_1_bci_ds,
    img_1_ai_ds,
    img_1_l4i_ds,
    img_1_fti_ds,
]
cow_r_1, cow_c_1 = slice(0, 80), slice(20, 100)
cow_r_2, cow_c_2 = slice(0, 40), slice(10, 50)


# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(3 * ncols, 4 * nrows), layout="constrained")
for i in range(ncols):
    axs[0, i].imshow(images[i], cmap="gray", vmin=0, vmax=255)
    axs[0, i].set(title=titles[i], xticks=[], yticks=[])
    if i == 0:
        axs[0, i].add_patch(
            patches.Rectangle(
                (cow_c_1.start, cow_r_1.start),
                cow_c_1.stop - cow_c_1.start,
                cow_r_1.stop - cow_r_1.start,
                linewidth=2,
                edgecolor="r",
                facecolor="none",
            )
        )
        axs[1, i].imshow(images[i][cow_r_1, cow_c_1], cmap="gray", vmin=0, vmax=255)
    else:
        axs[0, i].add_patch(
            patches.Rectangle(
                (cow_c_2.start, cow_r_2.start),
                cow_c_2.stop - cow_c_2.start,
                cow_r_2.stop - cow_r_2.start,
                linewidth=2,
                edgecolor="r",
                facecolor="none",
            )
        )
        axs[1, i].imshow(images[i][cow_r_2, cow_c_2], cmap="gray", vmin=0, vmax=255)
    axs[1, i].set(title="[zoom]", xticks=[], yticks=[])

plt.show()

### <a id='toc3_5_2_'></a>[Up Scaling](#toc0_)


In [ ]:
img_1_nni_us = cv2.resize(img_1_roi, dsize=(844, 844), interpolation=cv2.INTER_NEAREST_EXACT)
img_1_bli_us = cv2.resize(img_1_roi, dsize=(844, 844), interpolation=cv2.INTER_LINEAR)
img_1_bci_us = cv2.resize(img_1_roi, dsize=(844, 844), interpolation=cv2.INTER_CUBIC)
img_1_ai_us = cv2.resize(img_1_roi, dsize=(844, 844), interpolation=cv2.INTER_AREA)
img_1_l4i_us = cv2.resize(img_1_roi, dsize=(844, 844), interpolation=cv2.INTER_LANCZOS4)
img_1_fti_us = fourier_transform_interpolation(img_1_roi, 844, 844)

In [ ]:
titles = [
    f"{img_1.shape} Original",
    f"{img_1_nni_us.shape} nearest",
    f"{img_1_bli_us.shape} bilinear",
    f"{img_1_bci_us.shape} bicubic",
    f"{img_1_ai_us.shape} area",
    f"{img_1_l4i_us.shape} lanczos",
    f"{img_1_fti_us.shape} fourier transform",
]
images = [
    img_1_roi,
    img_1_nni_us,
    img_1_bli_us,
    img_1_bci_us,
    img_1_ai_us,
    img_1_l4i_us,
    img_1_fti_us,
]
cow_r_1, cow_c_1 = slice(0, 80), slice(20, 100)
cow_r_2, cow_c_2 = slice(0, 320), slice(80, 400)


# plot
nrows, ncols = 2, len(images)
fig, axs = plt.subplots(nrows, ncols, figsize=(3 * ncols, 4 * nrows), layout="constrained")
for i in range(ncols):
    axs[0, i].imshow(images[i], cmap="gray", vmin=0, vmax=255)
    axs[0, i].set(title=titles[i], xticks=[], yticks=[])
    if i == 0:
        axs[0, i].add_patch(
            patches.Rectangle(
                (cow_c_1.start, cow_r_1.start),
                cow_c_1.stop - cow_c_1.start,
                cow_r_1.stop - cow_r_1.start,
                linewidth=2,
                edgecolor="r",
                facecolor="none",
            )
        )
        axs[1, i].imshow(images[i][cow_r_1, cow_c_1], cmap="gray", vmin=0, vmax=255)
    else:
        axs[0, i].add_patch(
            patches.Rectangle(
                (cow_c_2.start, cow_r_2.start),
                cow_c_2.stop - cow_c_2.start,
                cow_r_2.stop - cow_r_2.start,
                linewidth=2,
                edgecolor="r",
                facecolor="none",
            )
        )
        axs[1, i].imshow(images[i][cow_r_2, cow_c_2], cmap="gray", vmin=0, vmax=255)
    axs[1, i].set(title="[zoom]", xticks=[], yticks=[])

plt.show()

## <a id='toc3_6_'></a>[Learnable / AI-based Interpolation](#toc0_)

- Classical interpolation methods (nearest, bilinear, bicubic, etc.) rely on fixed mathematical formulas.
- In contrast, **learnable interpolation** uses neural networks to estimate pixel values, allowing the model to adapt to the data and generate higher-quality results.
- This section explores how AI can perform interpolation and super-resolution.


### <a id='toc3_6_1_'></a>[Super-Resolution with CNNs](#toc0_)

- Convolutional Neural Networks (CNNs) can learn to upscale low-resolution images.

✍️ **Key points:**

- The network learns a mapping from low-res to high-res images.
- Typical architectures: [**SRCNN**](https://arxiv.org/abs/1501.00092), [**VDSR**](https://arxiv.org/abs/1511.04587), [**ESPCN**](https://arxiv.org/abs/1609.05158).
- Works well for natural images where classical interpolation introduces blur.

📝 **Docs**:

- Super-resolution Using an Efficient Sub-Pixel CNN: [github.com/pytorch/examples/tree/main/super_resolution](https://github.com/pytorch/examples/tree/main/super_resolution)


### <a id='toc3_6_2_'></a>[Super-Resolution with GANs](#toc0_)

- Generative Adversarial Networks (GANs) can enhance super-resolution by generating **photorealistic high-resolution images**:
  - [**SRGAN**](https://arxiv.org/abs/1609.04802), [**Real-ESRGAN**](https://arxiv.org/abs/2107.10833) are popular examples.
  - GANs introduce high-frequency textures that classical methods cannot recover.
  - They are especially effective for perceptual quality, though PSNR may be slightly lower.

📝 **Docs**:

- Hands-on SRGAN Implementation: [github.com/xinntao/ESRGAN](https://github.com/xinntao/ESRGAN)


### <a id='toc3_6_3_'></a>[Fourier / Frequency Domain Super-Resolution](#toc0_)

- Some networks operate in the **frequency domain**
  - They learn to reconstruct high-frequency components lost in downsampling.  
  - Useful for tasks like MRI or satellite imaging where frequency information is important.  
  - Connects nicely to classical FFT-based interpolation: AI learns which frequencies to restore rather than relying on zero-padding.

📝 **Docs**:

- Fourier-Guided Attention Upsampling for Image Super-Resolution: [arxiv.org/html/2508.10616v2](https://arxiv.org/html/2508.10616v2)


### <a id='toc3_6_4_'></a>[Learned Upsampling Layers in CNNs](#toc0_)

- Neural networks often use **upsampling layers** instead of classical interpolation:  
  - Common choices: nearest, bilinear, bicubic, or **learned kernels**.  
  - Examples: `nn.Upsample` in PyTorch, **transposed convolutions**, **sub-pixel convolution**.  

📝 **Docs**:

- PyTorch Upsampling Documentation: [docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)
